In [2]:
import pandas as pd
import requests
from polars.dependencies import pandas

adresse = "88 avenue verdier"
url_ban_example = f'https://api-adresse.data.gouv.fr/search/?q={adresse.replace(" ", "+")}&postcode=92120'
requests.get(url_ban_example)

<Response [200]>

In [3]:
req = requests.get(url_ban_example)
localisation_insee = req.json()
localisation_insee

{'type': 'FeatureCollection',
 'features': [{'type': 'Feature',
   'geometry': {'type': 'Point', 'coordinates': [2.309144, 48.81622]},
   'properties': {'label': '88 Avenue Verdier 92120 Montrouge',
    'score': 0.9734299999999999,
    'housenumber': '88',
    'id': '92049_9625_00088',
    'banId': '92dd3c4a-6703-423d-bf09-fc0412fb4f89',
    'name': '88 Avenue Verdier',
    'postcode': '92120',
    'citycode': '92049',
    'x': 649270.67,
    'y': 6857572.24,
    'city': 'Montrouge',
    'context': '92, Hauts-de-Seine, Île-de-France',
    'type': 'housenumber',
    'importance': 0.70773,
    'street': 'Avenue Verdier',
    '_type': 'address'}}],
 'query': '88 avenue verdier'}

In [4]:
localisation_insee.get('features')[0].get('properties')

{'label': '88 Avenue Verdier 92120 Montrouge',
 'score': 0.9734299999999999,
 'housenumber': '88',
 'id': '92049_9625_00088',
 'banId': '92dd3c4a-6703-423d-bf09-fc0412fb4f89',
 'name': '88 Avenue Verdier',
 'postcode': '92120',
 'citycode': '92049',
 'x': 649270.67,
 'y': 6857572.24,
 'city': 'Montrouge',
 'context': '92, Hauts-de-Seine, Île-de-France',
 'type': 'housenumber',
 'importance': 0.70773,
 'street': 'Avenue Verdier',
 '_type': 'address'}

In [5]:
# # Exercice 1:
adresse = "88 avenue verdier"
url_ban_example = f'https://api-adresse.data.gouv.fr/search/?q={adresse.replace(" ", "+")}'
req = requests.get(url_ban_example)
req.json()

{'type': 'FeatureCollection',
 'features': [{'type': 'Feature',
   'geometry': {'type': 'Point', 'coordinates': [2.309144, 48.81622]},
   'properties': {'label': '88 Avenue Verdier 92120 Montrouge',
    'score': 0.9734299999999999,
    'housenumber': '88',
    'id': '92049_9625_00088',
    'banId': '92dd3c4a-6703-423d-bf09-fc0412fb4f89',
    'name': '88 Avenue Verdier',
    'postcode': '92120',
    'citycode': '92049',
    'x': 649270.67,
    'y': 6857572.24,
    'city': 'Montrouge',
    'context': '92, Hauts-de-Seine, Île-de-France',
    'type': 'housenumber',
    'importance': 0.70773,
    'street': 'Avenue Verdier',
    '_type': 'address'}},
  {'type': 'Feature',
   'geometry': {'type': 'Point', 'coordinates': [-2.403652, 47.285572]},
   'properties': {'label': 'Avenue Verdier 44500 La Baule-Escoublac',
    'score': 0.7194127272727273,
    'id': '44055_3690',
    'name': 'Avenue Verdier',
    'postcode': '44500',
    'citycode': '44055',
    'x': 291884.83,
    'y': 6701220.48,
    

In [9]:
# # Avec code postal Montrouge:
postcode = 92120
url_ban_example = f'https://api-adresse.data.gouv.fr/search/?q={adresse.replace(" ", "+")}&postcode={postcode}'
req = requests.get(url_ban_example)
# req.json()
prop = req.json().get('features')[0].get('properties') 
x, y = prop.get('x'), prop.get('y') 
print(f'x: {x}, y: {y}')

x: 649270.67, y: 6857572.24


In [13]:
# # Représentation sur une carte:
# # Carte des stades avec Folium:
import pandas as pd
import geopandas as gpd
import folium

gdf = gpd.GeoDataFrame(data=pd.DataFrame({'ville': ['Montrouge']}), geometry=gpd.points_from_xy([x], [y]), crs='EPSG:2154')
gdf.to_crs(epsg=4326, inplace=True)
gdf['longitude'] = gdf.geometry.x
gdf['latitude'] = gdf.geometry.y

center = gdf[['latitude', 'longitude']].mean().values.tolist()
# sw = gdf[['latitude', 'longitude']].min().values.tolist()
# ne = gdf[['latitude', 'longitude']].max().values.tolist()

m = folium.Map(location = center, tiles='CartoDB.Positron') # CartoDB.Positron, openstreetmap 

# I can add marker one by one on the map
for i in range(0,len(gdf)):
    folium.Marker(
        [gdf.iloc[i]['latitude'], gdf.iloc[i]['longitude']],
        popup=gdf.iloc[i]['ville']
    ).add_to(m)

# m.fit_bounds([sw, ne])
m